In [1]:
# ============================================================
# D3 — Branch C: Deterministic Normalisation
# 0. Imports and frozen experimental configuration
# ============================================================
!pip -q install pymupdf

import json
import hashlib
import re
import sys
import platform
import unicodedata
from collections import Counter
from datetime import datetime
from pathlib import Path

import fitz
from google.colab import files

DOCUMENT_ID = "D3"
DOCUMENT_NAME = "Occupational Employment and Wages — May 2024"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

EXPECTED_SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = "240f41978ab36001450941f8d74ca7d58b0c865db6ffb6f72d3fcdc2c438a317"
EXPECTED_PAGE_COUNT = 23

SOURCE_PAGE_START = 1
SOURCE_PAGE_END = 5
REFERENCE_SCOPE_PAGES = list(range(SOURCE_PAGE_START, SOURCE_PAGE_END + 1))

EXPECTED_RECORD_COUNT = 70
REFERENCE_PERIOD = "May 2024"

EXPECTED_FIELDS = [
    "Section",
    "Indicator",
    "Occupation or Group",
    "Value",
    "Unit",
    "Reference Period"
]

ALLOWED_UNITS = [
    "workers",
    "million workers",
    "percent",
    "USD"
]

EXPECTED_SCOPE_MARKERS = [
    "OCCUPATIONAL EMPLOYMENT AND WAGES",
    "Production occupations",
    "Architecture and engineering occupations",
    "Building and grounds cleaning and maintenance occupations",
    "Largest occupations",
    "Public sector occupations",
    "May 2024"
]

NARRATIVE_SECTION_HEADINGS = [
    "Production occupations",
    "Architecture and engineering occupations",
    "Building and grounds cleaning and maintenance occupations",
    "Largest occupations",
    "Public sector occupations",
    "Changes to the Occupational Employment and Wage Statistics (OEWS) Data",
    "Introduction of New Metropolitan and Nonmetropolitan Area Definitions",
    "Suspension of Publication of Colorado Occupational Employment and Wage Statistics"
]

OUTPUT_DIR = Path("outputs_D3_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configured:", DOCUMENT_ID, BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected source pages:", EXPECTED_PAGE_COUNT)
print("Fixed Stage 1 scope:", REFERENCE_SCOPE_PAGES)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 35.9 MB/s eta 0:00:00
Configured: D3 C
Parent branch: B
Expected source pages: 23
Fixed Stage 1 scope: [1, 2, 3, 4, 5]


In [2]:
# ============================================================
# 1. Upload original D3 PDF and required Branch B artefacts
# ============================================================
# Upload exactly:
#   1) original D3 PDF
#   2) D3_branch_B_structural_markdown.md
#   3) D3_branch_B_conversion_integrity.json

uploaded = files.upload()
names = list(uploaded.keys())

pdf_files = [Path(f) for f in names if f.lower().endswith(".pdf")]
md_files = [Path(f) for f in names if f.lower().endswith(".md")]
json_files = [Path(f) for f in names if f.lower().endswith(".json")]

if len(pdf_files) != 1 or len(md_files) != 1 or len(json_files) != 1:
    raise ValueError(
        "Upload exactly one D3 PDF, one Branch B Markdown file, "
        "and one Branch B conversion-integrity JSON file."
    )

SOURCE_FILE = pdf_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_FILE.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D3_branch_B_conversion_integrity.json to D3_branch_B_conversion_integrity.json
Saving D3_branch_B_structural_markdown.md to D3_branch_B_structural_markdown.md
Saving D3 - ocwage.pdf to D3 - ocwage.pdf
Source: D3 - ocwage.pdf
Branch B representation: D3_branch_B_structural_markdown.md
Branch B integrity: D3_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B integrity
# ============================================================
def sha256_file(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_FILE.suffix.lower() != EXPECTED_SOURCE_FORMAT:
    raise ValueError(
        f"Expected {EXPECTED_SOURCE_FORMAT}; received {SOURCE_FILE.suffix}."
    )

SOURCE_SHA256 = sha256_file(SOURCE_FILE)
SOURCE_HASH_MATCH = SOURCE_SHA256 == EXPECTED_SOURCE_SHA256

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded PDF does not match the frozen D3 source identity."
    )

with open(BRANCH_B_CHECK_PATH, "r", encoding="utf-8") as f:
    branch_b_check = json.load(f)

if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError("The Branch B integrity file belongs to another document.")

if branch_b_check.get("branch") != "B":
    raise ValueError("The uploaded integrity file is not from Branch B.")

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "The Branch B parent representation was generated from a different source."
    )

if not branch_b_check.get("conversion_integrity_passed", False):
    raise ValueError(
        "Branch B parent representation did not pass conversion integrity."
    )

SOURCE_B_MARKDOWN = BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError("Uploaded Branch B Markdown is empty.")

SOURCE_B_SHA256 = sha256_text(SOURCE_B_MARKDOWN)

print("Frozen D3 source verified.")
print("Branch B conversion integrity verified.")
print("Uploaded Branch B SHA-256:", SOURCE_B_SHA256)


Frozen D3 source verified.
Branch B conversion integrity verified.
Uploaded Branch B SHA-256: 274f2778afc60da656519bd6740ccffabfb998b5a9cc0f4191ece97b61c8468d


In [4]:
# ============================================================
# 3. Reproduce the exact Branch B structural state
# ============================================================
pdf_document = fitz.open(SOURCE_FILE)
PAGE_COUNT = len(pdf_document)

if PAGE_COUNT != EXPECTED_PAGE_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_PAGE_COUNT} pages; observed {PAGE_COUNT}."
    )

page_character_counts = []
for page_number in range(1, PAGE_COUNT + 1):
    text = pdf_document[page_number - 1].get_text("text")
    page_character_counts.append(len(text.strip()))

MACHINE_READABLE = all(count > 0 for count in page_character_counts)

if not MACHINE_READABLE:
    raise ValueError(
        "D3 is expected to contain machine-readable text. OCR is not used."
    )

page_blocks = {}
block_audit = []

for page_number in range(1, PAGE_COUNT + 1):
    page = pdf_document[page_number - 1]
    blocks = page.get_text("blocks", sort=True)

    cleaned_page_blocks = []

    for block in blocks:
        x0, y0, x1, y1, text, block_number, block_type = block[:7]

        if not text or not text.strip():
            continue

        item = {
            "page": page_number,
            "block_number": int(block_number),
            "block_type": int(block_type),
            "x0": float(x0),
            "y0": float(y0),
            "x1": float(x1),
            "y1": float(y1),
            "text": text
        }

        cleaned_page_blocks.append(item)
        block_audit.append(item)

    page_blocks[page_number] = cleaned_page_blocks


def is_isolated_page_marker(line):
    # Frozen Branch B behaviour: retain all source lines.
    return False


def canonical_heading_if_exact(line):
    candidate = line.strip().rstrip(":")
    for heading in NARRATIVE_SECTION_HEADINGS:
        if candidate.casefold() == heading.casefold():
            return heading
    return None


def is_bullet_line(line):
    stripped = line.lstrip()
    return stripped.startswith("•") or stripped.startswith("\u2022")


def remove_bullet_marker(line):
    stripped = line.lstrip()
    if stripped.startswith("•"):
        return stripped[1:].lstrip()
    return line.strip()


markdown_lines = [
    "# Occupational Employment and Wages — May 2024",
    "",
    "> Structural conversion of the complete 23-page source PDF.",
    "> The extraction scope remains the fixed Stage 1 headline narrative scope on pages 1–5.",
    ""
]

retained_source_lines = []
excluded_source_lines = []

for page_number in range(1, PAGE_COUNT + 1):
    markdown_lines.append(f"## Source Page {page_number}")
    markdown_lines.append("")

    for block in page_blocks[page_number]:
        for raw_line in block["text"].splitlines():
            line = raw_line.strip()

            if not line:
                continue

            if is_isolated_page_marker(line):
                excluded_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "reason": "isolated_page_marker"
                })
                continue

            heading = canonical_heading_if_exact(line)

            if heading is not None:
                markdown_lines.append(f"### {heading}")
                markdown_lines.append("")
                retained_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "representation": "recognised_heading"
                })
                continue

            if is_bullet_line(line):
                bullet_text = remove_bullet_marker(line)
                markdown_lines.append(f"- {bullet_text}")
                retained_source_lines.append({
                    "page": page_number,
                    "text": line,
                    "representation": "bullet"
                })
                continue

            markdown_lines.append(line)
            retained_source_lines.append({
                "page": page_number,
                "text": line,
                "representation": "source_text"
            })

        markdown_lines.append("")

REPRODUCED_BRANCH_B_MARKDOWN = "\n".join(markdown_lines).rstrip() + "\n"
REPRODUCED_B_SHA256 = sha256_text(REPRODUCED_BRANCH_B_MARKDOWN)

print("Reproduced Branch B characters:", len(REPRODUCED_BRANCH_B_MARKDOWN))
print("Reproduced Branch B SHA-256:", REPRODUCED_B_SHA256)


Reproduced Branch B characters: 213589
Reproduced Branch B SHA-256: 274f2778afc60da656519bd6740ccffabfb998b5a9cc0f4191ece97b61c8468d


In [5]:
# ============================================================
# 4. Verify exact Branch B parent equivalence
# ============================================================
PARENT_EQUIVALENCE_PASSED = (
    REPRODUCED_BRANCH_B_MARKDOWN == SOURCE_B_MARKDOWN
)

parent_check = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,
    "source_sha256": SOURCE_SHA256,
    "branch_B_conversion_integrity_passed":
        bool(branch_b_check.get("conversion_integrity_passed", False)),
    "uploaded_branch_B_sha256": SOURCE_B_SHA256,
    "reproduced_branch_B_sha256": REPRODUCED_B_SHA256,
    "branch_B_representation_exactly_reproduced":
        PARENT_EQUIVALENCE_PASSED,
    "source_page_count": PAGE_COUNT,
    "retained_source_line_count": len(retained_source_lines),
    "excluded_source_line_count": len(excluded_source_lines),
    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D3_branch_C_parent_B_equivalence_check.json"
)

PARENT_CHECK_PATH.write_text(
    json.dumps(parent_check, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(parent_check, indent=2, ensure_ascii=False))

if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "Uploaded Branch B Markdown does not exactly match the "
        "structural representation reproduced from the frozen D3 PDF."
    )


{
  "document_id": "D3",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "240f41978ab36001450941f8d74ca7d58b0c865db6ffb6f72d3fcdc2c438a317",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "274f2778afc60da656519bd6740ccffabfb998b5a9cc0f4191ece97b61c8468d",
  "reproduced_branch_B_sha256": "274f2778afc60da656519bd6740ccffabfb998b5a9cc0f4191ece97b61c8468d",
  "branch_B_representation_exactly_reproduced": true,
  "source_page_count": 23,
  "retained_source_line_count": 6149,
  "excluded_source_line_count": 0,
  "parent_equivalence_passed": true
}


In [6]:
# ============================================================
# 5. Define deterministic Branch C normalisation functions
# ============================================================
UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f", "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "–": "-",
    "—": "-",
    "−": "-",
    "‐": "-"
}


def normalise_unicode(text):
    return unicodedata.normalize("NFKC", text)


def normalise_unicode_spaces(text):
    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(character, " ")
    return text


def normalise_apostrophes(text):
    for original, replacement in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(original, replacement)
    return text


def normalise_dashes(text):
    for original, replacement in DASH_REPLACEMENTS.items():
        text = text.replace(original, replacement)
    return text


def normalise_horizontal_whitespace(text):
    """
    Collapse spaces/tabs within each line while preserving every
    non-empty line and all page/Markdown boundaries.
    """
    lines = []
    for line in text.splitlines():
        lines.append(re.sub(r"[ \t]+", " ", line).strip())
    return "\n".join(lines)


def normalise_currency_spacing(text):
    """
    Formatting-only normalisation:
        $ 50,090 -> $50,090
    """
    return re.sub(r"\$\s+(?=\d)", "$", text)


def normalise_numeric_comma_spacing(text):
    """
    Formatting-only normalisation:
        685, 140 -> 685,140
    """
    return re.sub(
        r"(?<=\d)\s*,\s*(?=\d{3}\b)",
        ",",
        text
    )


def normalise_markdown_syntax_spacing(text):
    """
    Standardise spacing after existing Markdown headings and bullets.
    No new headings/bullets are inferred.
    """
    lines = []
    for line in text.splitlines():
        if re.match(r"^#{1,6}\s*", line):
            m = re.match(r"^(#{1,6})\s*(.*)$", line)
            line = f"{m.group(1)} {m.group(2).strip()}"
        elif re.match(r"^-\s*", line):
            line = "- " + re.sub(r"^-\s*", "", line).strip()
        lines.append(line)
    return "\n".join(lines)


def normalise_blank_lines(text):
    return re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"


def normalise_representation(text):
    """
    D3 Branch C deterministic normalisation.

    Deliberately NOT applied:
    - paragraph merging;
    - line-break hyphenation repair;
    - semantic label rewriting;
    - unit remapping;
    - numerical rescaling;
    - precision changes;
    - page removal;
    - content reconstruction.

    This conservative design ensures that Branch C changes notation
    and formatting only while retaining the complete Branch B content.
    """
    text = normalise_unicode(text)
    text = normalise_unicode_spaces(text)
    text = normalise_apostrophes(text)
    text = normalise_dashes(text)
    text = normalise_horizontal_whitespace(text)
    text = normalise_currency_spacing(text)
    text = normalise_numeric_comma_spacing(text)
    text = normalise_markdown_syntax_spacing(text)
    text = normalise_blank_lines(text)
    return text


In [7]:
# ============================================================
# 6. Apply Branch C deterministic normalisation
# ============================================================
NORMALISED_MARKDOWN = normalise_representation(
    SOURCE_B_MARKDOWN
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError("Branch C normalisation produced an empty representation.")

print("Branch B characters:", len(SOURCE_B_MARKDOWN))
print("Branch C characters:", len(NORMALISED_MARKDOWN))
print("Branch B non-empty lines:",
      sum(bool(line.strip()) for line in SOURCE_B_MARKDOWN.splitlines()))
print("Branch C non-empty lines:",
      sum(bool(line.strip()) for line in NORMALISED_MARKDOWN.splitlines()))

print("\nBeginning of Branch C representation:\n")
print(NORMALISED_MARKDOWN[:2500])


Branch B characters: 213589
Branch C characters: 213546
Branch B non-empty lines: 6175
Branch C non-empty lines: 6175

Beginning of Branch C representation:

# Occupational Employment and Wages - May 2024

> Structural conversion of the complete 23-page source PDF.
> The extraction scope remains the fixed Stage 1 headline narrative scope on pages 1-5.

## Source Page 1

For release 10:00 a.m. (ET) Wednesday, April 2, 2025
USDL-25-0451
Technical information: (202) 691-6569 • oewsinfo@bls.gov • www.bls.gov/oes
Media contact: (202) 691-5902 • PressOffice@bls.gov

OCCUPATIONAL EMPLOYMENT AND WAGES - MAY 2024
Production occupations had employment of 8.7 million in May 2024, representing 5.7 percent of total
national employment, the U.S. Bureau of Labor Statistics reported today. The largest production
occupations were miscellaneous assemblers and fabricators (1.5 million) and first-line supervisors of
production and operating workers (685,140). (See chart 1.) The annual mean wage across all

In [9]:
# ============================================================
# 7. Verify Branch C normalisation integrity
# ============================================================

# ------------------------------------------------------------
# IMPORTANT D3 INTEGRITY PRINCIPLE
# ------------------------------------------------------------
#
# Branch C is allowed to apply Unicode NFKC normalisation.
#
# In D3, the source contains many superscript footnote markers
# such as:
#
#     ²
#     ³
#
# Unicode NFKC converts these to:
#
#     2
#     3
#
# Therefore, a generic regex that counts every standalone integer
# before and after normalisation will incorrectly interpret these
# representational changes as newly added numerical values.
#
# For D3, numerical integrity is therefore assessed using
# value-bearing numerical patterns relevant to the document:
#
#   - currency values
#   - percentages
#   - values expressed in millions
#   - comma-separated quantitative values
#
# In addition, the complete content-line sequence is verified
# after applying the SAME deterministic Branch C normalisation
# to the Branch B side.
#
# This ensures that Branch C changes representation only and
# does not add, remove, reorder, calculate, round, or reconstruct
# source information.
# ------------------------------------------------------------


# ------------------------------------------------------------
# 1. Preserve all source-page boundaries
# ------------------------------------------------------------

page_boundary_checks = {
    str(page): (
        f"## Source Page {page}"
        in NORMALISED_MARKDOWN
    )
    for page in range(
        1,
        EXPECTED_PAGE_COUNT + 1
    )
}

all_page_boundaries_preserved = all(
    page_boundary_checks.values()
)


# ------------------------------------------------------------
# 2. Preserve all fixed Stage 1 scope markers
# ------------------------------------------------------------

scope_marker_checks = {
    marker: (
        marker.casefold()
        in NORMALISED_MARKDOWN.casefold()
    )
    for marker in EXPECTED_SCOPE_MARKERS
}

all_scope_markers_preserved = all(
    scope_marker_checks.values()
)


# ------------------------------------------------------------
# 3. Verify complete content-line sequence
# ------------------------------------------------------------
#
# Raw-string equality is inappropriate because Branch C is
# explicitly allowed to normalise Unicode, punctuation and
# whitespace.
#
# Instead, apply the SAME Branch C normalisation to the parent
# Branch B representation and compare the resulting non-empty
# line sequence.
# ------------------------------------------------------------

def canonical_nonempty_lines(text):

    canonical_text = normalise_representation(
        text
    )

    return [
        line
        for line in canonical_text.splitlines()
        if line.strip()
    ]


parent_canonical_lines = (
    canonical_nonempty_lines(
        SOURCE_B_MARKDOWN
    )
)

branch_c_canonical_lines = (
    canonical_nonempty_lines(
        NORMALISED_MARKDOWN
    )
)


content_line_sequence_preserved = (
    parent_canonical_lines
    == branch_c_canonical_lines
)


# ------------------------------------------------------------
# 4. Verify non-empty line count
# ------------------------------------------------------------

parent_nonempty_count = sum(
    bool(line.strip())
    for line
    in SOURCE_B_MARKDOWN.splitlines()
)

branch_c_nonempty_count = sum(
    bool(line.strip())
    for line
    in NORMALISED_MARKDOWN.splitlines()
)

nonempty_line_count_preserved = (
    parent_nonempty_count
    == branch_c_nonempty_count
)


# ------------------------------------------------------------
# 5. Numerical preservation
# ------------------------------------------------------------
#
# Do NOT use a generic standalone-integer pattern here.
#
# D3 contains superscript footnotes (² and ³), which NFKC
# legitimately converts to ordinary characters (2 and 3).
#
# Instead, verify the value-bearing numerical constructions
# relevant to this document.
# ------------------------------------------------------------

NUMERIC_PATTERNS = {

    # Examples:
    # $50,090
    # $ 50,090
    # $103,980
    "currency":
        r"\$\s*\d[\d,\s]*(?:\.\d+)?",

    # Examples:
    # 5.7 percent
    # 44.0 percent
    "percent":
        r"\b\d+(?:\.\d+)?\s*percent\b",

    # Examples:
    # 8.7 million
    # 1.5 million
    "million":
        r"\b\d+(?:\.\d+)?\s*million\b",

    # Examples:
    # 685,140
    # 154,187,380
    #
    # Permits spaces introduced around commas in the source
    # representation and canonicalises them before comparison.
    "comma_separated_number":
        r"\b\d{1,3}(?:\s*,\s*\d{3})+\b"
}


def canonicalise_numeric_token(token):
    """
    Canonicalise representation-level variation in a numerical
    token without changing its numerical meaning.
    """

    token = normalise_unicode(
        token
    )

    token = normalise_unicode_spaces(
        token
    )

    # Collapse horizontal whitespace
    token = re.sub(
        r"[ \t]+",
        " ",
        token
    )

    # $ 50,090 -> $50,090
    token = re.sub(
        r"\$\s+",
        "$",
        token
    )

    # 685, 140 -> 685,140
    token = re.sub(
        r"(?<=\d)\s*,\s*(?=\d{3}\b)",
        ",",
        token
    )

    return token.strip().casefold()


numeric_preservation = {}


for label, pattern in NUMERIC_PATTERNS.items():

    before_tokens = [
        canonicalise_numeric_token(token)

        for token in re.findall(
            pattern,
            SOURCE_B_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]

    after_tokens = [
        canonicalise_numeric_token(token)

        for token in re.findall(
            pattern,
            NORMALISED_MARKDOWN,
            flags=re.IGNORECASE
        )
    ]


    before_counter = Counter(
        before_tokens
    )

    after_counter = Counter(
        after_tokens
    )


    missing_tokens = list(
        (
            before_counter
            - after_counter
        ).elements()
    )

    added_tokens = list(
        (
            after_counter
            - before_counter
        ).elements()
    )


    pattern_passed = (
        len(missing_tokens) == 0
        and len(added_tokens) == 0
    )


    numeric_preservation[label] = {

        "count_before":
            len(before_tokens),

        "count_after":
            len(after_tokens),

        "missing_token_count":
            len(missing_tokens),

        "added_token_count":
            len(added_tokens),

        "passed":
            pattern_passed
    }


numeric_tokens_preserved = all(
    result["passed"]
    for result
    in numeric_preservation.values()
)


# ------------------------------------------------------------
# 6. Overall Branch C normalisation integrity
# ------------------------------------------------------------

normalisation_integrity_passed = bool(

    PARENT_EQUIVALENCE_PASSED

    and all_page_boundaries_preserved

    and all_scope_markers_preserved

    and content_line_sequence_preserved

    and nonempty_line_count_preserved

    and numeric_tokens_preserved
)


# ------------------------------------------------------------
# 7. Build detailed integrity report
# ------------------------------------------------------------

normalisation_check = {

    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,


    # --------------------------------------------------------
    # Page preservation
    # --------------------------------------------------------

    "source_page_count":
        EXPECTED_PAGE_COUNT,

    "all_source_page_boundaries_preserved":
        all_page_boundaries_preserved,

    "page_boundary_checks":
        page_boundary_checks,


    # --------------------------------------------------------
    # Stage 1 extraction scope preservation
    # --------------------------------------------------------

    "fixed_stage1_scope_pages": [
        SOURCE_PAGE_START,
        SOURCE_PAGE_END
    ],

    "scope_marker_checks":
        scope_marker_checks,

    "all_scope_markers_preserved":
        all_scope_markers_preserved,


    # --------------------------------------------------------
    # Complete-content preservation
    # --------------------------------------------------------

    "parent_nonempty_line_count":
        parent_nonempty_count,

    "branch_C_nonempty_line_count":
        branch_c_nonempty_count,

    "nonempty_line_count_preserved":
        nonempty_line_count_preserved,

    "content_line_sequence_preserved":
        content_line_sequence_preserved,


    # --------------------------------------------------------
    # Numerical integrity
    # --------------------------------------------------------

    "numeric_token_preservation":
        numeric_preservation,

    "numeric_tokens_preserved":
        numeric_tokens_preserved,

    "generic_integer_check_excluded":
        True,

    "generic_integer_check_exclusion_reason":
        (
            "D3 contains superscript footnote markers such as "
            "² and ³. Unicode NFKC legitimately converts these "
            "to ordinary digits, so generic standalone-integer "
            "counts are not a valid numerical-integrity test."
        ),


    # --------------------------------------------------------
    # Applied Branch C transformations
    # --------------------------------------------------------

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_standardisation_applied":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "currency_spacing_standardisation_applied":
        True,

    "numeric_comma_spacing_standardisation_applied":
        True,

    "markdown_syntax_spacing_standardisation_applied":
        True,

    "blank_line_standardisation_applied":
        True,


    # --------------------------------------------------------
    # Transformations explicitly NOT applied
    # --------------------------------------------------------

    "paragraph_line_merging_applied":
        False,

    "line_break_hyphenation_repair_applied":
        False,

    "semantic_label_rewriting_applied":
        False,

    "unit_semantic_remapping_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "out_of_scope_pages_removed":
        False,

    "value_modification_applied":
        False,

    "value_rounding_applied":
        False,

    "derived_calculation_applied":
        False,

    "rounded_million_values_expanded":
        False,

    "reference_values_used_for_transformation":
        False,


    # --------------------------------------------------------
    # Final decision
    # --------------------------------------------------------

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


# ------------------------------------------------------------
# 8. Save integrity report
# ------------------------------------------------------------

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR
    / "D3_branch_C_normalisation_check.json"
)

NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. Display results
# ------------------------------------------------------------

print(
    json.dumps(
        normalisation_check,
        indent=2,
        ensure_ascii=False
    )
)


# ------------------------------------------------------------
# 10. Stop only for a genuine integrity failure
# ------------------------------------------------------------

if not normalisation_integrity_passed:

    raise ValueError(
        "D3 Branch C normalisation-integrity checks failed. "
        "Inspect page, content-order, scope-marker and "
        "value-bearing numerical preservation diagnostics."
    )

{
  "document_id": "D3",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "source_page_count": 23,
  "all_source_page_boundaries_preserved": true,
  "page_boundary_checks": {
    "1": true,
    "2": true,
    "3": true,
    "4": true,
    "5": true,
    "6": true,
    "7": true,
    "8": true,
    "9": true,
    "10": true,
    "11": true,
    "12": true,
    "13": true,
    "14": true,
    "15": true,
    "16": true,
    "17": true,
    "18": true,
    "19": true,
    "20": true,
    "21": true,
    "22": true,
    "23": true
  },
  "fixed_stage1_scope_pages": [
    1,
    5
  ],
  "scope_marker_checks": {
    "OCCUPATIONAL EMPLOYMENT AND WAGES": true,
    "Production occupations": true,
    "Architecture and engineering occupations": true,
    "Building and grounds cleaning and maintenance occupations": true,
    "Largest occupations": true,
    "Public sector occupations": true,
    "May 2024": true
  },
  "all_scope_markers_preserved": true,
  "parent

In [11]:
# ============================================================
# 8. Save Branch C representation
# ============================================================
REPRESENTATION_PATH = (
    OUTPUT_DIR / "D3_branch_C_normalised_markdown.md"
)

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(REPRESENTATION_PATH)

print("Saved:", REPRESENTATION_PATH.name)
print("Representation SHA-256:", REPRESENTATION_SHA256)


Saved: D3_branch_C_normalised_markdown.md
Representation SHA-256: 94e7373836e7ad752c837b8b2760a90f8c1d646bd68efe38b1ab16965ed786c7


In [12]:
# ============================================================
# 9. Define the fixed extraction schema
# ============================================================
EXPECTED_OUTPUT_STRUCTURE = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "records": [
        {
            "Section": None,
            "Indicator": None,
            "Occupation or Group": None,
            "Value": None,
            "Unit": None,
            "Reference Period": None
        }
    ]
}

print(json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
))


{
  "document_id": "D3",
  "branch": "C",
  "records": [
    {
      "Section": null,
      "Indicator": null,
      "Occupation or Group": null,
      "Value": null,
      "Unit": null,
      "Reference Period": null
    }
  ]
}


In [13]:
# ============================================================
# 10. Operationalise the fixed Stage 1 extraction task
# ============================================================
# The instruction strength mirrors Branch B.
# The expected 70-record count is NOT disclosed to the model.

EXTRACTION_TASK = """
You are an information extraction assistant.

Extract every explicitly stated occupational statistic from the headline
narrative sections corresponding to source pages 1–5 of the attached
deterministically normalised Markdown document.

Return one record for every statistic represented within the defined
scope.

For each record, extract:

- Section
- Indicator
- Occupation or Group
- Value
- Unit
- Reference Period

Scope and extraction rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Use only the headline narrative sections corresponding to source
  pages 1–5.
- Extract only information explicitly supported by the document.
- Do not extract the release identifier, release date, contact details,
  website addresses or other publication metadata.
- Do not extract general programme-description counts.
- Do not extract the Technical Note.
- Do not extract the full multi-page Table 1.
- Do not create separate observations from charts when the same values
  are already stated in the narrative.
- Do not include headings without numerical observations as records.
- Do not calculate, infer, reconstruct, aggregate, correct or invent
  any value.
- Do not increase the precision of rounded values.
- Preserve rounded employment values in the reported scale. For example,
  "8.7 million" must be returned as Value 8.7 with Unit
  "million workers".
- Return exact employment counts as numerical values with Unit
  "workers".
- Return employment shares and concentration values as numerical
  values with Unit "percent".
- Return annual mean wages as numerical values with Unit "USD".
- Preserve the occupation, group, industry or location wording used in
  the headline narrative.
- Use "May 2024" as the Reference Period for every record.
- Return Value as a numerical value, not as formatted text.
- Use null only when a requested value is not available.
- Verify that only the defined headline narrative scope has been
  processed.
- Verify that every explicitly stated occupational statistic within
  that scope has been processed.
- Verify that rounded values remain in their reported scale.
- Return only valid JSON.
- Do not include explanations before or after the JSON.
- Keep the exact field names defined in the schema.
""".strip()

FULL_PROMPT = f"""
{EXTRACTION_TASK}

Expected JSON schema:
{json.dumps(
    EXPECTED_OUTPUT_STRUCTURE,
    indent=2,
    ensure_ascii=False
)}

The complete deterministically normalised Markdown representation of the
23-page source document is attached.

The extraction scope remains restricted to the headline narrative
sections corresponding to source pages 1–5.

Return only the JSON object.
""".strip()

PROMPT_PATH = OUTPUT_DIR / "D3_branch_C_prompt.txt"
PROMPT_PATH.write_text(FULL_PROMPT, encoding="utf-8")
PROMPT_SHA256 = sha256_file(PROMPT_PATH)

print(FULL_PROMPT)
print("Prompt SHA-256:", PROMPT_SHA256)


You are an information extraction assistant.

Extract every explicitly stated occupational statistic from the headline
narrative sections corresponding to source pages 1–5 of the attached
deterministically normalised Markdown document.

Return one record for every statistic represented within the defined
scope.

For each record, extract:

- Section
- Indicator
- Occupation or Group
- Value
- Unit
- Reference Period

Scope and extraction rules:

- Treat the attached deterministically normalised Markdown document as
  the only source of information.
- Use only the headline narrative sections corresponding to source
  pages 1–5.
- Extract only information explicitly supported by the document.
- Do not extract the release identifier, release date, contact details,
  website addresses or other publication metadata.
- Do not extract general programme-description counts.
- Do not extract the Technical Note.
- Do not extract the full multi-page Table 1.
- Do not create separate observations fr

In [14]:
# ============================================================
# 11. Create Branch C representation metadata
# ============================================================
REPRESENTATION_METADATA = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "parent_branch": PARENT_BRANCH,

    "source_file": SOURCE_FILE.name,
    "source_format": SOURCE_FILE.suffix.lower(),
    "source_sha256": SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,
    "parent_B_representation_sha256":
        SOURCE_B_SHA256,
    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "representation_type":
        "Complete Branch B structural Markdown with deterministic normalisation",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,

    "source_pages":
        EXPECTED_PAGE_COUNT,
    "normalised_pages":
        EXPECTED_PAGE_COUNT,
    "fixed_extraction_scope_pages":
        [SOURCE_PAGE_START, SOURCE_PAGE_END],

    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "out_of_scope_content_retained": True,

    "normalisation_operations": [
        "Unicode NFKC normalisation",
        "Unicode-space standardisation",
        "apostrophe standardisation",
        "dash standardisation",
        "horizontal whitespace normalisation",
        "currency-spacing standardisation",
        "numeric comma-spacing standardisation",
        "existing Markdown syntax spacing standardisation",
        "blank-line standardisation"
    ],

    "operations_explicitly_not_applied": [
        "OCR",
        "Page cropping",
        "Removal of pages 6–23",
        "Paragraph line merging",
        "Line-break hyphenation repair",
        "Semantic label harmonisation",
        "Semantic unit remapping",
        "Value calculation or repair",
        "Value precision increase",
        "Expansion of rounded million-worker values",
        "Reference-value-guided transformation"
    ],

    "reference_values_used_for_transformation": False,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"]
}

REP_METADATA_PATH = (
    OUTPUT_DIR / "D3_branch_C_representation_metadata.json"
)

REP_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(
    REPRESENTATION_METADATA,
    indent=2,
    ensure_ascii=False
))


{
  "document_id": "D3",
  "branch": "C",
  "parent_branch": "B",
  "source_file": "D3 - ocwage.pdf",
  "source_format": ".pdf",
  "source_sha256": "240f41978ab36001450941f8d74ca7d58b0c865db6ffb6f72d3fcdc2c438a317",
  "parent_B_representation_file": "D3_branch_B_structural_markdown.md",
  "parent_B_representation_sha256": "274f2778afc60da656519bd6740ccffabfb998b5a9cc0f4191ece97b61c8468d",
  "parent_B_equivalence_passed": true,
  "representation_type": "Complete Branch B structural Markdown with deterministic normalisation",
  "representation_file": "D3_branch_C_normalised_markdown.md",
  "representation_sha256": "94e7373836e7ad752c837b8b2760a90f8c1d646bd68efe38b1ab16965ed786c7",
  "source_pages": 23,
  "normalised_pages": 23,
  "fixed_extraction_scope_pages": [
    1,
    5
  ],
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "ocr_applied": false,
  "page_cropping_applied": false,
  "out_of_scope_content_retained": true,
  "normalisation_oper

In [15]:
# ============================================================
# 12. Create experiment metadata and pre-extraction control
# ============================================================
EXPERIMENT_METADATA = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,

    "source_file": SOURCE_FILE.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified":
        SOURCE_HASH_MATCH and PAGE_COUNT == EXPECTED_PAGE_COUNT,

    "input_representation":
        "Complete deterministically normalised structural Markdown",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "direct_document_ingestion": False,
    "structural_conversion_applied": True,
    "structural_conversion_inherited_from_branch_B": True,
    "normalisation_applied": True,
    "ocr_applied": False,
    "page_cropping_applied": False,
    "out_of_scope_content_retained": True,

    "fixed_extraction_scope_pages":
        [SOURCE_PAGE_START, SOURCE_PAGE_END],

    "reference_values_disclosed_to_model": False,
    "expected_record_count_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,

    "manual_response_repair_permitted": False,
    "expected_output_format": "JSON",

    "prompt_file": PROMPT_PATH.name,
    "prompt_sha256": PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",
    "model": "GPT-5.5",

    "created_at": datetime.now().isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),

    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using Branch A-frozen comparison rules"
}

METADATA_PATH = (
    OUTPUT_DIR / "D3_branch_C_experiment_metadata.json"
)

METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

PRECHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "source_identity_verified":
        SOURCE_HASH_MATCH,
    "source_page_count_verified":
        PAGE_COUNT == EXPECTED_PAGE_COUNT,
    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],
    "all_23_pages_retained":
        all(page_boundary_checks.values()),
    "fixed_scope_markers_preserved":
        all(scope_marker_checks.values()),
    "representation_exists":
        REPRESENTATION_PATH.exists(),
    "prompt_exists":
        PROMPT_PATH.exists(),
    "expected_record_count_disclosed_to_model":
        False,
    "reference_values_used_for_transformation":
        False,
    "ready_for_independent_llm_execution": bool(
        SOURCE_HASH_MATCH
        and PAGE_COUNT == EXPECTED_PAGE_COUNT
        and PARENT_EQUIVALENCE_PASSED
        and normalisation_check["normalisation_integrity_passed"]
        and REPRESENTATION_PATH.exists()
        and PROMPT_PATH.exists()
    )
}

PRECHECK_PATH = (
    OUTPUT_DIR / "D3_branch_C_pre_extraction_check.json"
)

PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(PRECHECK, indent=2, ensure_ascii=False))

if not PRECHECK["ready_for_independent_llm_execution"]:
    raise ValueError(
        "D3 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D3",
  "branch": "C",
  "source_identity_verified": true,
  "source_page_count_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "all_23_pages_retained": true,
  "fixed_scope_markers_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [16]:
# ============================================================
# 13. Download pre-extraction Branch C artefacts
# ============================================================
for p in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH
]:
    files.download(p)

print(
    "\nIndependent execution instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D3_branch_C_normalised_markdown.md.\n"
    "3. Submit D3_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the PDF, Branch B artefacts, Stage 1 reference values, "
    "or previous extraction outputs.\n"
    "5. Do not manually correct, repair, or regenerate the model response.\n"
    "6. Save the complete response exactly as returned in a plain-text file."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent execution instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D3_branch_C_normalised_markdown.md.
3. Submit D3_branch_C_prompt.txt exactly once.
4. Do not upload the PDF, Branch B artefacts, Stage 1 reference values, or previous extraction outputs.
5. Do not manually correct, repair, or regenerate the model response.
6. Save the complete response exactly as returned in a plain-text file.


In [17]:
# ============================================================
# 14. Upload and preserve the complete raw Branch C response
# ============================================================
uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw Branch C response text file."
    )

RAW_RESPONSE_SOURCE = Path(next(iter(uploaded_response)))
RAW_RESPONSE_TEXT = RAW_RESPONSE_SOURCE.read_text(encoding="utf-8")

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D3_branch_C_raw_response.txt"
)

RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)

RAW_RESPONSE_SHA256 = sha256_file(RAW_RESPONSE_PATH)

print("Raw Branch C response preserved unchanged.")
print("Raw response SHA-256:", RAW_RESPONSE_SHA256)


Saving D3_branch_C_raw_response.txt to D3_branch_C_raw_response.txt
Raw Branch C response preserved unchanged.
Raw response SHA-256: 2cf25b2f2b35f0354021b339fbdd56db7d972820ff6edc9a2a834cc92c49261c


In [18]:
# ============================================================
# 15. Parse the raw response without repair
# ============================================================
valid_json = True
json_error = None
raw_extraction = None

try:
    raw_extraction = json.loads(RAW_RESPONSE_TEXT)
except json.JSONDecodeError as exc:
    valid_json = False
    json_error = str(exc)

print("Valid JSON:", valid_json)

if json_error:
    print("JSON parsing error:", json_error)


Valid JSON: True


In [19]:
# ============================================================
# 16. Validate top-level output structure
# ============================================================
top_level_object_valid = (
    valid_json and isinstance(raw_extraction, dict)
)

document_id_present = (
    top_level_object_valid
    and "document_id" in raw_extraction
)

document_id_correct = (
    document_id_present
    and raw_extraction.get("document_id") == DOCUMENT_ID
)

branch_present = (
    top_level_object_valid
    and "branch" in raw_extraction
)

branch_correct = (
    branch_present
    and raw_extraction.get("branch") == BRANCH
)

records_present = (
    top_level_object_valid
    and "records" in raw_extraction
)

records_is_list = (
    records_present
    and isinstance(raw_extraction.get("records"), list)
)

records = (
    raw_extraction["records"]
    if records_is_list
    else []
)

TOP_LEVEL_CHECK = {
    "top_level_object_valid":
        bool(top_level_object_valid),
    "document_id_present":
        bool(document_id_present),
    "document_id_correct":
        bool(document_id_correct),
    "branch_present":
        bool(branch_present),
    "branch_correct":
        bool(branch_correct),
    "records_present":
        bool(records_present),
    "records_is_list":
        bool(records_is_list)
}

print(json.dumps(TOP_LEVEL_CHECK, indent=2))


{
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true
}


In [20]:
# ============================================================
# 17. Validate record schemas, field types, units and periods
# ============================================================
record_structure_issues = []
field_type_issues = []
unit_issues = []
reference_period_issues = []

for record_index, record in enumerate(records):

    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(
        expected_fields - actual_fields
    )

    additional_fields = sorted(
        actual_fields - expected_fields
    )

    if missing_fields or additional_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "additional_fields": additional_fields
        })

    for field in [
        "Section",
        "Indicator",
        "Occupation or Group",
        "Unit",
        "Reference Period"
    ]:
        value = record.get(field)

        if (
            value is not None
            and not isinstance(value, str)
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    value = record.get("Value")

    if (
        value is not None
        and (
            isinstance(value, bool)
            or not isinstance(value, (int, float))
        )
    ):
        field_type_issues.append({
            "record_index": record_index,
            "field": "Value",
            "observed_type": type(value).__name__,
            "observed_value": value
        })

    unit = record.get("Unit")

    if (
        isinstance(unit, str)
        and unit not in ALLOWED_UNITS
    ):
        unit_issues.append({
            "record_index": record_index,
            "unit": unit
        })

    period = record.get("Reference Period")

    if (
        isinstance(period, str)
        and period != REFERENCE_PERIOD
    ):
        reference_period_issues.append({
            "record_index": record_index,
            "reference_period": period
        })

records_with_type_issues = len({
    issue["record_index"]
    for issue in field_type_issues
})

print("Record structure issues:", len(record_structure_issues))
print("Records with type issues:", records_with_type_issues)
print("Unexpected units:", len(unit_issues))
print("Unexpected reference periods:", len(reference_period_issues))


Record structure issues: 0
Records with type issues: 0
Unexpected units: 0
Unexpected reference periods: 0


In [21]:
# ============================================================
# 18. Scope, duplicate and excluded-content diagnostics
# ============================================================
record_count = len(records)
scope_complete = (
    record_count == EXPECTED_RECORD_COUNT
)

def complete_record_key(record):
    if not isinstance(record, dict):
        return None

    return tuple(
        record.get(field)
        for field in EXPECTED_FIELDS
    )

keys = [
    complete_record_key(record)
    for record in records
    if isinstance(record, dict)
]

duplicate_record_key_count = (
    len(keys) - len(set(keys))
)

excluded_content_markers = [
    "technical note",
    "table 1",
    "media contact",
    "press office",
    "release identifier"
]

excluded_content_issues = []

for record_index, record in enumerate(records):

    if not isinstance(record, dict):
        continue

    descriptive_text = " ".join(
        str(record.get(field, ""))
        for field in [
            "Section",
            "Indicator",
            "Occupation or Group"
        ]
    ).casefold()

    markers = [
        marker
        for marker in excluded_content_markers
        if marker in descriptive_text
    ]

    if markers:
        excluded_content_issues.append({
            "record_index": record_index,
            "matched_markers": markers
        })

missing_values_by_field = {
    field: sum(
        1
        for record in records
        if (
            not isinstance(record, dict)
            or field not in record
            or record.get(field) is None
        )
    )
    for field in EXPECTED_FIELDS
}

print("Expected reference-scope records:", EXPECTED_RECORD_COUNT)
print("Observed extracted records:", record_count)
print("Scope complete:", scope_complete)
print("Duplicate full-record keys:", duplicate_record_key_count)
print("Excluded-content issues:", len(excluded_content_issues))
print("Missing values:", missing_values_by_field)


Expected reference-scope records: 70
Observed extracted records: 74
Scope complete: False
Duplicate full-record keys: 0
Excluded-content issues: 0
Missing values: {'Section': 0, 'Indicator': 0, 'Occupation or Group': 0, 'Value': 0, 'Unit': 0, 'Reference Period': 0}


In [22]:
# ============================================================
# 19. Keep schema validity separate from scope completeness
# ============================================================
# The expected record count is NOT part of schema validity.
# A response can be structurally valid while missing or adding
# observations; those are extraction/scope outcomes.

schema_validity = bool(
    valid_json
    and top_level_object_valid
    and document_id_present
    and document_id_correct
    and branch_present
    and branch_correct
    and records_present
    and records_is_list
    and len(record_structure_issues) == 0
    and records_with_type_issues == 0
    and len(unit_issues) == 0
    and len(reference_period_issues) == 0
    and duplicate_record_key_count == 0
    and len(excluded_content_issues) == 0
)

STRUCTURE_CHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,

    "valid_json": bool(valid_json),
    "json_error": json_error,

    **TOP_LEVEL_CHECK,

    "schema_validity":
        schema_validity,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        int(record_count),
    "scope_complete":
        bool(scope_complete),

    "records_with_structure_issues":
        len(record_structure_issues),
    "record_structure_issues":
        record_structure_issues,

    "records_with_type_issues":
        records_with_type_issues,
    "field_type_issues":
        field_type_issues,

    "records_with_unexpected_units":
        len(unit_issues),
    "unit_issues":
        unit_issues,

    "records_with_unexpected_reference_periods":
        len(reference_period_issues),
    "reference_period_issues":
        reference_period_issues,

    "duplicate_record_key_count":
        int(duplicate_record_key_count),

    "excluded_content_issue_count":
        len(excluded_content_issues),
    "excluded_content_issues":
        excluded_content_issues,

    "missing_values_by_field":
        missing_values_by_field
}

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D3_branch_C_structure_check.json"
)

STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(
    STRUCTURE_CHECK,
    indent=2,
    ensure_ascii=False
))


{
  "document_id": "D3",
  "branch": "C",
  "valid_json": true,
  "json_error": null,
  "top_level_object_valid": true,
  "document_id_present": true,
  "document_id_correct": true,
  "branch_present": true,
  "branch_correct": true,
  "records_present": true,
  "records_is_list": true,
  "schema_validity": true,
  "expected_record_count": 70,
  "observed_record_count": 74,
  "scope_complete": false,
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "records_with_unexpected_units": 0,
  "unit_issues": [],
  "records_with_unexpected_reference_periods": 0,
  "reference_period_issues": [],
  "duplicate_record_key_count": 0,
  "excluded_content_issue_count": 0,
  "excluded_content_issues": [],
  "missing_values_by_field": {
    "Section": 0,
    "Indicator": 0,
    "Occupation or Group": 0,
    "Value": 0,
    "Unit": 0,
    "Reference Period": 0
  }
}


In [23]:
# ============================================================
# 20. Preserve parsed extraction only when JSON is valid
# ============================================================
PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D3_branch_C_parsed_extraction.json"
)

if valid_json:
    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            raw_extraction,
            indent=2,
            ensure_ascii=False
        ),
        encoding="utf-8"
    )

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:
    print(
        "No parsed extraction created because the preserved "
        "raw response is not valid JSON."
    )


Parsed extraction saved: D3_branch_C_parsed_extraction.json


In [24]:
# ============================================================
# 21. Create final Branch C experiment summary
# ============================================================
EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "parent_branch": PARENT_BRANCH,

    "source_file": SOURCE_FILE.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified":
        SOURCE_HASH_MATCH and PAGE_COUNT == EXPECTED_PAGE_COUNT,

    "input_representation":
        "Complete deterministically normalised structural Markdown",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":
        normalisation_check["normalisation_integrity_passed"],

    "structural_conversion_inherited_from_branch_B":
        True,
    "normalisation_applied":
        True,
    "ocr_applied":
        False,
    "page_cropping_applied":
        False,
    "out_of_scope_content_retained":
        True,

    "fixed_extraction_scope_pages":
        [SOURCE_PAGE_START, SOURCE_PAGE_END],

    "reference_values_used_for_transformation":
        False,
    "expected_record_count_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,
    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(valid_json),
    "schema_validity":
        bool(schema_validity),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        int(record_count),
    "scope_complete":
        bool(scope_complete),

    "records_with_structure_issues":
        len(record_structure_issues),
    "records_with_type_issues":
        records_with_type_issues,
    "records_with_unexpected_units":
        len(unit_issues),
    "records_with_unexpected_reference_periods":
        len(reference_period_issues),
    "duplicate_record_key_count":
        int(duplicate_record_key_count),
    "excluded_content_issue_count":
        len(excluded_content_issues),

    "parsed_extraction_created":
        bool(valid_json),

    "accuracy_validation_completed":
        False,

    "validation_status":
        "Pending Stage 4 Branch C validation against the fixed Stage 1 "
        "reference dataset using Branch A-frozen comparison rules"
}

SUMMARY_PATH = (
    OUTPUT_DIR / "D3_branch_C_experiment_summary.json"
)

SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_SUMMARY,
    indent=2,
    ensure_ascii=False
))


{
  "document_id": "D3",
  "document_name": "Occupational Employment and Wages — May 2024",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D3 - ocwage.pdf",
  "source_sha256": "240f41978ab36001450941f8d74ca7d58b0c865db6ffb6f72d3fcdc2c438a317",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D3_branch_C_normalised_markdown.md",
  "representation_sha256": "94e7373836e7ad752c837b8b2760a90f8c1d646bd68efe38b1ab16965ed786c7",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "ocr_applied": false,
  "page_cropping_applied": false,
  "out_of_scope_content_retained": true,
  "fixed_extraction_scope_pages": [
    1,
    5
  ],
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_model": f

In [25]:
# ============================================================
# 22. Final artefact inventory and download
# ============================================================
artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    REP_METADATA_PATH,
    METADATA_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    SUMMARY_PATH
]

if valid_json:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )

print("Final D3 Branch C artefacts:")

for p in artefacts:
    print(
        "-",
        p.name,
        "| exists:",
        p.exists()
    )

for p in artefacts:
    if p.exists():
        files.download(p)


Final D3 Branch C artefacts:
- D3_branch_C_parent_B_equivalence_check.json | exists: True
- D3_branch_C_normalisation_check.json | exists: True
- D3_branch_C_normalised_markdown.md | exists: True
- D3_branch_C_prompt.txt | exists: True
- D3_branch_C_representation_metadata.json | exists: True
- D3_branch_C_experiment_metadata.json | exists: True
- D3_branch_C_pre_extraction_check.json | exists: True
- D3_branch_C_raw_response.txt | exists: True
- D3_branch_C_structure_check.json | exists: True
- D3_branch_C_experiment_summary.json | exists: True
- D3_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>